[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/casos-de-uso/03-resumen-documentos.ipynb)

# 03 — Resumen automático de documentos

> **Bloque:** Casos de uso | **Nivel:** Práctico
>
> Notebook complementario al tutorial [03-resumen-documentos.md](../../casos-de-uso/03-resumen-documentos.md)

Practicamos las estrategias de resumen: directo, map-reduce y estructurado.

In [ ]:
# %pip install anthropic python-dotenv

import anthropic
import json
import time
from tqdm.notebook import tqdm

client = anthropic.Anthropic()
MODELO_RAPIDO = 'claude-haiku-4-5-20251001'
MODELO_BUENO  = 'claude-sonnet-4-6'
print('✓ Listo')

## Texto de ejemplo

Usaremos un artículo simulado sobre IA para practicar.

In [ ]:
TEXTO_CORTO = """
La inteligencia artificial generativa ha transformado radicalmente el mundo tecnológico en los
últimos años. Modelos como GPT-4 de OpenAI y Claude de Anthropic han demostrado capacidades
sorprendentes en generación de texto, código y análisis de información. Las empresas de todos
los sectores están integrando estas tecnologías para automatizar tareas, mejorar la atención
al cliente y acelerar la innovación. El mercado de IA generativa superó los 40.000 millones de
dólares en 2024 y se espera que alcance los 200.000 millones en 2030. Sin embargo, este avance
también plantea importantes preguntas éticas sobre la privacidad, los sesgos algorítmicos y el
impacto en el mercado laboral. Los expertos coinciden en que la regulación responsable es clave
para maximizar los beneficios mientras se minimizan los riesgos.
"""

# Texto largo simulado (repetimos párrafos para simular documento extenso)
TEXTO_LARGO = (TEXTO_CORTO.strip() + '\n\n') * 8  # ~8 párrafos

print(f'Texto corto: {len(TEXTO_CORTO.split())} palabras')
print(f'Texto largo: {len(TEXTO_LARGO.split())} palabras')

## 1. Resumen directo (texto corto)

In [ ]:
def resumir(texto: str, estilo: str = 'ejecutivo', modelo: str = MODELO_RAPIDO) -> str:
    estilos = {
        'ejecutivo':   'Resume de forma ejecutiva: idea principal, puntos clave y conclusión.',
        'bullets':     'Resume en 5 bullets (•) concisos.',
        'divulgativo': 'Resume en lenguaje muy sencillo para público no especializado.',
    }
    prompt = f"""{estilos.get(estilo, estilos['ejecutivo'])}
Máximo 3 frases o 5 bullets. Solo el resumen, sin introducción.

<documento>
{texto}
</documento>"""
    r = client.messages.create(
        model=modelo, max_tokens=300, temperature=0.3,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.content[0].text

for estilo in ['ejecutivo', 'bullets', 'divulgativo']:
    print(f'\n=== {estilo.upper()} ===')
    print(resumir(TEXTO_CORTO, estilo=estilo))

## 2. División en chunks

In [ ]:
def dividir_chunks(texto: str, tamano: int = 150, solapamiento: int = 20) -> list:
    palabras = texto.split()
    chunks, inicio = [], 0
    while inicio < len(palabras):
        fin = min(inicio + tamano, len(palabras))
        chunks.append(' '.join(palabras[inicio:fin]))
        if fin == len(palabras): break
        inicio = fin - solapamiento
    return chunks

chunks = dividir_chunks(TEXTO_LARGO, tamano=150)
print(f'Documento de {len(TEXTO_LARGO.split())} palabras → {len(chunks)} chunks')
print(f'\nPrimer chunk ({len(chunks[0].split())} palabras):')
print(chunks[0][:200] + '...')

## 3. Estrategia Map-Reduce

In [ ]:
def map_reduce(texto: str) -> dict:
    chunks = dividir_chunks(texto)
    print(f'MAP: resumiendo {len(chunks)} chunks...')

    # MAP: resumir cada chunk
    resumenes = []
    for i, chunk in enumerate(tqdm(chunks, desc='Chunks')):
        r = client.messages.create(
            model=MODELO_RAPIDO, max_tokens=150, temperature=0.0,
            messages=[{'role': 'user', 'content':
                f'Resume en 2 frases los puntos clave de este fragmento ({i+1}/{len(chunks)}):\n{chunk}'}]
        )
        resumenes.append(r.content[0].text)
        time.sleep(0.2)

    # REDUCE: combinar
    print('\nREDUCE: combinando resúmenes...')
    resumen_parciales = '\n\n'.join([f'Fragmento {i+1}:\n{r}' for i, r in enumerate(resumenes)])
    r_final = client.messages.create(
        model=MODELO_BUENO, max_tokens=400, temperature=0.3,
        messages=[{'role': 'user', 'content':
            f'Crea un resumen final cohesivo de 2-3 párrafos a partir de estos resúmenes parciales:\n\n{resumen_parciales}'}]
    )
    return {'resumen': r_final.content[0].text, 'num_chunks': len(chunks), 'parciales': resumenes}

resultado = map_reduce(TEXTO_LARGO)
print('\n=== RESUMEN FINAL (Map-Reduce) ===')
print(resultado['resumen'])

## 4. Resumen estructurado en JSON

In [ ]:
def resumir_estructurado(texto: str) -> dict:
    prompt = f"""Analiza el siguiente texto y devuelve SOLO un JSON con este formato:
{{
  "titulo_sugerido": "",
  "resumen_ejecutivo": "",
  "puntos_clave": ["...", "...", "..."],
  "datos_numericos": ["..."],
  "conclusion": "",
  "palabras_clave": ["...", "...", "..."]
}}

<documento>
{texto}
</documento>"""
    r = client.messages.create(
        model=MODELO_BUENO, max_tokens=600, temperature=0.0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    raw = r.content[0].text.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    return json.loads(raw)

datos = resumir_estructurado(TEXTO_CORTO)
print(json.dumps(datos, ensure_ascii=False, indent=2))

## 5. Comparativa: directo vs map-reduce

In [ ]:
import matplotlib.pyplot as plt

# Comparativa conceptual de costes y tiempo según estrategia
estrategias = ['Directo\n(<3K palabras)', 'Map-Reduce\n(3K-50K)', 'Refine\n(coherencia alta)']
velocidad   = [5, 3, 2]   # 1=lento, 5=rápido
coherencia  = [5, 3, 5]
coste       = [1, 3, 4]   # 1=barato, 5=caro

x = range(len(estrategias))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w for i in x], velocidad,  w, label='Velocidad',  color='#2ECC71', alpha=0.85)
ax.bar([i     for i in x], coherencia, w, label='Coherencia', color='#3498DB', alpha=0.85)
ax.bar([i + w for i in x], coste,      w, label='Coste',      color='#E74C3C', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(estrategias)
ax.set_ylim(0, 6)
ax.set_ylabel('Puntuación (1=bajo, 5=alto)')
ax.set_title('Comparativa de estrategias de resumen')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Prueba con tu propio texto

In [ ]:
# Pega aquí tu propio texto o carga un fichero
mi_texto = """
Escribe o pega aquí el texto que quieres resumir...
"""

if len(mi_texto.split()) > 10:  # Evitar ejecutar con el placeholder
    print('=== Resumen ejecutivo ===')
    print(resumir(mi_texto, estilo='ejecutivo'))
    print('\n=== Bullets ===')
    print(resumir(mi_texto, estilo='bullets'))
else:
    print('Reemplaza mi_texto con tu propio contenido para probarlo.')

---
**Tutorial relacionado:** [03-resumen-documentos.md](../../casos-de-uso/03-resumen-documentos.md)